# Dataset import and first modelling

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

Dataset from: https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data

The original dataset has been modified for this hands-on session.

In [ ]:
# Import training dataset and train the simplest model
df_train = pd.read_csv("../data/train.csv")
X_train = df_train["GrLivArea"].values.reshape(-1, 1)
y_train = df_train["SalePrice"]
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
# Import test set and predict on it
df_test = pd.read_csv("../data/X_test.csv")
X_test = df_test["GrLivArea"]
y_pred = model.predict(X_test.values.reshape(-1, 1))

# Uncertainty band quantification

In [ ]:
# Set the lower and upper bounds as 10% of the prediction, as an example
lower_bound = y_pred * 0.9
upper_bound = y_pred * 1.1

In [ ]:
# Create a DataFrame for submission
df_results = pd.DataFrame({
    "y_pred": y_pred,
    "lower_bound": lower_bound,
    "upper_bound": upper_bound
})

# Compute the mean relative uncertainty band (definition invented for this hands-on)
MRUB = np.mean((upper_bound-lower_bound)/y_pred)
print(f"MRUB: {MRUB:.2f}")

# Submit results

In [ ]:
# Define functions for leaderboard submission

import requests
import urllib3

urllib3.disable_warnings()

leaderboard_endpoint = "https://compute.datascience.ch/leaderboard-innovation/api/v1/leaderboard"
submit_endpoint = "https://compute.datascience.ch/leaderboard-innovation/api/v1/submit"

def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(leaderboard_endpoint, verify=False).json()
    return pd.DataFrame(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str) -> pd.Series:
    predictions = df.to_dict(orient="records")
    submission = {"predictions": predictions}
    response = requests.post(submit_endpoint, json=submission, verify=False, auth=(user, "1234"))
    if (response.status_code // 100) != 2:
        response.raise_for_status()
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code})")
    entry = response.json()
    return pd.Series(entry)

In [ ]:
# Submitions are ranked by the smallest MRUB if coverage >= 90%
get_leaderboard()

In [ ]:
# Uncomment the following line to submit your predictions, anche add your name
#submit(df_results, user="YOUR_NAME_HERE")